# EvidenceIQ — Building the Database

In 01_eda.ipynb we explored the raw file and found the traps.
Here we turn those findings into a clean database that everyone on the team queries.

Why a database and not just a DataFrame?
EvidenceIQ answers questions by writing and running SQL. So the data has to live
somewhere SQL can reach it. We use DuckDB — it's a single file, needs no server,
and it can be opened read-only so the AI can never change the data.

Run this once. After that, everyone just connects to the database file.

Output: data/processed/evidenceiq.duckdb


## Step 1: Load the raw file

Same as the EDA notebook: two sheets, one per year. Takes about 45 seconds.


In [21]:
import pandas as pd
import duckdb
import os

RAW = "../data/raw/online_retail_II.xlsx"
DB  = "../data/processed/evidenceiq.duckdb"

sheet1 = pd.read_excel(RAW, sheet_name="Year 2009-2010")
sheet2 = pd.read_excel(RAW, sheet_name="Year 2010-2011")

df = pd.concat([sheet1, sheet2], ignore_index=True)
print("Rows loaded:", len(df))

Rows loaded: 1067371


## Step 2: Remove duplicates

Remember from the EDA: the two sheets overlap for the first nine days of
December 2010, so those rows appear twice. If we skip this step, December 2010
looks 51% bigger than it really is.


In [22]:
rows_before = len(df)
df = df.drop_duplicates()

print("Removed", rows_before - len(df), "duplicate rows")
print("Rows now:", len(df))

Removed 34335 duplicate rows
Rows now: 1033036


## Step 3: Remove the bad-debt adjustments

A few invoices start with the letter A. They are accounting entries, not sales...
and one of them is −£53,594, which would badly distort whichever month it lands in.

Let's look at them before deleting.


In [23]:
df["Invoice"] = df["Invoice"].astype(str)
df["StockCode"] = df["StockCode"].astype(str)

df[df["Invoice"].str.startswith("A")][["Invoice", "Description", "Quantity", "Price"]]

,Invoice,Description,Quantity,Price
179403,A506401,Adjust bad debt,1,-53594.36
276274,A516228,Adjust bad debt,1,-44031.79
403472,A528059,Adjust bad debt,1,-38925.87
825443,A563185,Adjust bad debt,1,11062.06
825444,A563186,Adjust bad debt,1,-11062.06
825445,A563187,Adjust bad debt,1,-11062.06


In [24]:
df = df[~df["Invoice"].str.startswith("A")]
print("Rows now:", len(df))

Rows now: 1033030


## Step 4: Rename the columns

The raw names have spaces and capitals (Customer ID), which are awkward in SQL.
We switch to lowercase with underscores so queries are easier to write — for us
and for the AI.


In [25]:
df = df.rename(columns={
    "Invoice":     "invoice_no",
    "StockCode":   "stock_code",
    "Description": "description",
    "Quantity":    "quantity",
    "InvoiceDate": "invoice_ts",
    "Price":       "unit_price",
    "Customer ID": "customer_id",
    "Country":     "country",
})

df.columns

Index(['invoice_no', 'stock_code', 'description', 'quantity', 'invoice_ts',
       'unit_price', 'customer_id', 'country'],
      dtype='object')

## Step 5: Add helpful columns

Rather than making the AI work these out every time, we calculate them once.

- revenue — quantity × price
- is_cancellation — invoices starting with C
- is_product — False for postage, fees and vouchers, which are not products
- is_outlier — the two freak orders that were cancelled minutes after being placed
- invoice_month — stored as text like "2011-11", which is much easier to
  write a WHERE clause against than a date function


In [26]:
NOT_PRODUCTS = ["DOT", "POST", "C2", "M", "m", "S", "B", "BANK CHARGES",
                "AMAZONFEE", "ADJUST", "ADJUST2", "PADS", "CRUK",
                "TEST001", "TEST002", "DCGSSGIRL", "DCGSSBOY"]

# Two orders placed and cancelled within minutes. They are real rows, but they
# are so large they distort any returns figure, so we flag them instead of deleting.
ODD_INVOICES = ["541431", "C541433", "581483", "C581484"]

df["revenue"]         = df["quantity"] * df["unit_price"]
df["is_cancellation"] = df["invoice_no"].str.startswith("C")
df["is_product"]      = ~df["stock_code"].isin(NOT_PRODUCTS)
df["is_outlier"]      = df["invoice_no"].isin(ODD_INVOICES)
df["invoice_date"]    = df["invoice_ts"].dt.date
df["invoice_month"]   = df["invoice_ts"].dt.to_period("M").astype(str)

df.head(3)

,invoice_no,stock_code,description,quantity,invoice_ts,unit_price,customer_id,country,revenue,is_cancellation,is_product,is_outlier,invoice_date,invoice_month
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,83.4,False,True,False,2009-12-01,2009-12
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0,False,True,False,2009-12-01,2009-12
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0,False,True,False,2009-12-01,2009-12


## Step 6: Check the numbers

These two numbers are our agreed baseline. If someone changes a cleaning rule,
this cell fails and we find out immediately — instead of discovering three weeks
later that two teammates have been quoting different revenue figures.


In [27]:
print("Rows: ", len(df))
print("Revenue: £{:,.2f}".format(df["revenue"].sum()))

assert len(df) == 1_033_030,                          "Row count changed!"
assert round(df["revenue"].sum(), 2) == 19_003_147.78, "Revenue changed!"

print("\nChecks passed.")

Rows:  1033030
Revenue: £19,003,147.78

Checks passed.


## Step 7: Create the database

We make one main table and three summary tables.

sales is the important one — every line item. The other three are convenience
tables so common questions don't need complicated SQL.


In [28]:
os.makedirs("../data/processed", exist_ok=True)
if os.path.exists(DB):
    os.remove(DB)          # always rebuild from scratch

con = duckdb.connect(DB)
con.execute("CREATE TABLE sales AS SELECT * FROM df")

print("sales rows:", con.execute("SELECT COUNT(*) FROM sales").fetchone()[0])

sales rows: 1033030


### dim_month: one row per month

This table holds the December warning. Instead of hoping the AI remembers that
December 2011 is incomplete, it can just look at the is_complete_month column.


In [29]:
con.execute('''
CREATE TABLE dim_month AS
SELECT
    invoice_month,
    COUNT(DISTINCT invoice_date) AS trading_days,
    MAX(invoice_date)            AS last_day,
    SUM(revenue)                 AS net_revenue,
    invoice_month <> (SELECT MAX(invoice_month) FROM sales) AS is_complete_month
FROM sales
GROUP BY invoice_month
ORDER BY invoice_month
''')

con.execute("SELECT * FROM dim_month ORDER BY invoice_month DESC LIMIT 5").df()

,invoice_month,trading_days,last_day,net_revenue,is_complete_month
0,2011-12,8,2011-12-09,432719.060,False
1,2011-11,26,2011-11-30,1456145.800,True
2,2011-10,26,2011-10-31,1069368.230,True
3,2011-09,26,2011-09-30,1017596.682,True
4,2011-08,26,2011-08-31,692448.520,True


December 2011 has 8 trading days and is_complete_month = False. Exactly what we wanted.


### dim_product: one row per product

We use mode(description) — the most common description — because the raw data
gives the same product code different names (one is literally
"reverse 21/5/10 adjustment").


In [30]:
con.execute('''
CREATE TABLE dim_product AS
SELECT
    stock_code,
    mode(description)                          AS description,
    SUM(quantity) FILTER (WHERE quantity > 0)  AS units_sold,
    SUM(revenue)  FILTER (WHERE quantity > 0)  AS gross_revenue
FROM sales
GROUP BY stock_code
''')

con.execute("SELECT * FROM dim_product ORDER BY gross_revenue DESC LIMIT 5").df()

,stock_code,description,units_sold,gross_revenue
0,M,Manual,9637.0,339599.81
1,22423,REGENCY CAKESTAND 3 TIER,26495.0,330590.32
2,DOT,DOTCOM POSTAGE,2920.0,309854.11
3,85123A,WHITE HANGING HEART T-LIGHT HOLDER,98208.0,257724.71
4,85099B,JUMBO BAG RED RETROSPOT,96764.0,180569.34


### dim_customer: one row per customer

Customers with no ID are excluded here (22.8% of rows). That's unavoidable —
but it means any answer about customers is missing part of the picture,
and EvidenceIQ has to say so.


In [31]:
con.execute('''
CREATE TABLE dim_customer AS
SELECT
    customer_id,
    mode(country)     AS country,
    MIN(invoice_date) AS first_order,
    MAX(invoice_date) AS last_order,
    COUNT(DISTINCT invoice_no) FILTER (WHERE NOT is_cancellation) AS orders,
    SUM(revenue)      AS net_revenue
FROM sales
WHERE customer_id IS NOT NULL
GROUP BY customer_id
''')

con.execute("SELECT * FROM dim_customer ORDER BY net_revenue DESC LIMIT 5").df()

,customer_id,country,first_order,last_order,orders,net_revenue
0,18102.0,United Kingdom,2009-12-01,2011-12-09,145,570380.61
1,14646.0,Netherlands,2009-12-02,2011-12-08,152,523342.07
2,14156.0,EIRE,2009-12-01,2011-11-30,156,296063.44
3,14911.0,EIRE,2009-12-01,2011-12-08,398,265757.91
4,17450.0,United Kingdom,2010-09-27,2011-12-01,51,231390.55


In [32]:
for table in ["sales", "dim_month", "dim_product", "dim_customer"]:
    n = con.execute("SELECT COUNT(*) FROM " + table).fetchone()[0]
    print("{:<15} {:>10,} rows".format(table, n))

con.close()
print("\nDatabase written to", DB)

sales            1,033,030 rows
dim_month               25 rows
dim_product          5,304 rows
dim_customer         5,942 rows

Database written to ../data/processed/evidenceiq.duckdb


## Step 8: Test it

Reopen the database read-only and check a few answers we already worked out by hand.


In [33]:
con = duckdb.connect(DB, read_only=True)

con.execute('''
SELECT ROUND(SUM(revenue), 2) AS november_2011
FROM sales
WHERE invoice_month = '2011-11'
''').df()

,november_2011
0,1456145.8


Expected £1,456,145.80.


In [34]:
con.execute('''
SELECT country, ROUND(SUM(revenue), 2) AS revenue
FROM sales
WHERE invoice_month LIKE '2011%'
  AND country <> 'United Kingdom'
GROUP BY country
ORDER BY revenue DESC
LIMIT 3
''').df()

,country,revenue
0,Netherlands,275877.06
1,EIRE,253963.43
2,Germany,206982.03


Expected the Netherlands at £275,877.06.


### The return rate: the same question, two answers

This is why the KPI definitions matter. Both queries are correct SQL.


In [35]:
con.execute('''
SELECT
    ROUND(100 * SUM(CASE WHEN quantity < 0 THEN -revenue END)
              / SUM(CASE WHEN quantity > 0 THEN  revenue END), 2) AS everything,

    ROUND(100 * SUM(CASE WHEN quantity < 0 AND is_product AND NOT is_outlier THEN -revenue END)
              / SUM(CASE WHEN quantity > 0 THEN  revenue END), 2) AS real_products_only
FROM sales
WHERE invoice_month LIKE '2011%'
''').df()

,everything,real_products_only
0,8.35,2.22


About 8% versus 2.36%. The difference is fees, manual adjustments
and two freak cancelled orders.

Neither number is wrong — but EvidenceIQ must always give the same one.
That is what docs/kpi_definitions.md is for.


### The AI cannot damage the data

Because we opened the database with read_only=True, DuckDB itself refuses to
write. We don't have to trust the AI to behave.


In [36]:
try:
    con.execute("DELETE FROM sales")
    print("Uh oh - the delete worked!")
except Exception as e:
    print("Blocked, as expected:")
    print(e)

Blocked, as expected:
Invalid Input Error: Cannot execute statement of type "DELETE" on database "evidenceiq" which is attached in read-only mode!


In [37]:
con.close()